# M1 实验队列 · Kaggle Runner

**用法**:Kaggle → New Notebook → File → Import Notebook 粘贴本文件(或从 GitHub `everest-an/M1` 的 `kaggle/kaggle_runner.ipynb` 导入)→ 右侧 Settings 选 **GPU T4** → Run All。

- 每个实验独立,跑完一个就把结果(`reasoning_depth.jsonl` + log)拷到 `/kaggle/working/`,**12h 会话被杀也不丢已完成的**。
- 会话结束后在 Output 面板下载 `m1_results.zip`,发回给 CC 入库。
- 断点续跑:把下面 `START` 改成上次完成的序号+1,重新 Run All。

**当前队列(2026-08-01)— 裁决 GQA 配额 + J-Space J1,grok 率协议**:
探针改用单环+课程混合任务(比固定难度可靠地 grok,per-k 评估信息量更大);每配置 3 seeds,报告各跳数准确率。

In [ ]:
import subprocess, os, shutil, time, glob

# ── 环境 ──
if not os.path.exists('/kaggle/working/M1'):
    subprocess.run(['git','clone','--depth=1','https://github.com/everest-an/M1.git','/kaggle/working/M1'], check=True)
os.chdir('/kaggle/working/M1')
subprocess.run(['git','pull','--ff-only'], check=False)
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# ── 实验队列(独立可重排;想换实验只改这里)──
BASE = ('python benchmarks/reasoning_depth.py --task pointer_chase --n_values 8 '
        '--mode fixed --skip_transformer --mix --difficulty 4 --steps 30000 ')
QUEUE = [
    # A. GQA 配额裁决(单环 mix 任务,g0 vs g2,3 seeds 一次跑完)
    BASE + '--seeds 0 1 2 --eval_depths 1 --n_global_heads 0 --tag kg-mix-g0',
    BASE + '--seeds 0 1 2 --eval_depths 1 --n_global_heads 2 --tag kg-mix-g2',
    # B. J-Space J1 工作区驻留 sweep(docs/JSPACE_DESIGN.md §2,已接线)
    BASE + '--seeds 0 1 2 --eval_depths 1 2 4 --workspace --tag kg-j1',
    # C. 补 Kaggle 上次没跑完的第四格(full MHA + quota,固定难度 2 对齐旧口径)
    BASE.replace('--mix ','') + '--difficulty 2 --seeds 0 --eval_depths 1 --full_mha --n_global_heads 2 --tag kg-mha-g2',
]
START = 0  # 断点续跑:改成上次完成的序号+1

def snapshot():
    for f in glob.glob('benchmarks/results/*.jsonl') + glob.glob('benchmarks/results/*.log'):
        shutil.copy(f, '/kaggle/working/')

for i, cmd in enumerate(QUEUE):
    if i < START: continue
    print(f'\n══════ [{i}] {cmd}\n', flush=True)
    t0 = time.time()
    r = subprocess.run(cmd.split(), capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0: print('STDERR:', r.stderr[-2000:])
    print(f'[{i}] 用时 {(time.time()-t0)/60:.1f} min, exit={r.returncode}', flush=True)
    snapshot()
print('\n全部完成')

In [ ]:
# ── 打包下载 ──
import shutil
shutil.make_archive('/kaggle/working/m1_results', 'zip', '/kaggle/working/M1/benchmarks/results')
print('下载 /kaggle/working/m1_results.zip,发回给 CC 入库')